# 📦 Funnel Analysis — E-Commerce Order Analytics


In [ ]:
# Uncomment if running for the first time
# !pip install -q seaborn scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

# Visual configuration
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

print("✅ Libraries ready")


## 📁 Upload CSV Files

In [ ]:
from google.colab import files
import io

print("Upload 6 CSV files: categories, customers, orders, order_items, products, returns")
uploaded = files.upload()

dataframes = {}
for fname, content in uploaded.items():
    key = fname.replace(".csv", "").lower()
    dataframes[key] = pd.read_csv(io.BytesIO(content))
    print(f"  ✓ {fname} — {len(dataframes[key]):,} rows")

required = ["categories", "customers", "orders", "order_items", "products", "returns"]
missing  = [r for r in required if r not in dataframes]
if missing:
    print(f"\n⚠️  Missing files: {missing}")
else:
    print("\n✅ All files loaded successfully")


## 🔧 Merge & Prepare Data

In [ ]:
categories  = dataframes["categories"]
customers   = dataframes["customers"]
orders      = dataframes["orders"]
order_items = dataframes["order_items"]
products    = dataframes["products"]
returns     = dataframes["returns"]

# --- Merge tables ---
oi = orders.merge(order_items, on="order_id", how="left")
oi = oi.merge(products[["product_id", "category_id"]], on="product_id", how="left")
oi = oi.merge(categories, on="category_id", how="left")

oi["revenue"] = oi["quantity"] * oi["unit_price"]

delivered_ids  = orders.loc[orders["order_status"] == "Delivered", "order_id"]
ret_with_order = returns.merge(
    order_items[["order_item_id", "order_id"]], on="order_item_id", how="left"
)
returned_order_ids = ret_with_order.loc[
    ret_with_order["order_id"].isin(delivered_ids), "order_id"
].unique()

oi["is_returned"] = oi["order_id"].isin(returned_order_ids).astype(int)

# --- Order-level summary ---
order_summary = (
    oi.groupby("order_id")
    .agg(
        customer_id    = ("customer_id",    "first"),
        order_status   = ("order_status",   "first"),
        payment_method = ("payment_method", "first"),
        total_revenue  = ("revenue",        "sum"),   # FIX #1
        total_items    = ("order_item_id",  "count"),
        is_returned    = ("is_returned",    "max"),
    )
    .reset_index()
)

print("✅ Data ready")
print(f"   order_summary : {order_summary.shape}")
print(f"   orders_items  : {oi.shape}")
order_summary.head(3)


## 📊 Key Metrics

In [ ]:
total_orders    = order_summary["order_id"].nunique()
delivered_count = (order_summary["order_status"] == "Delivered").sum()
returned_count  = int(order_summary["is_returned"].sum())
completed_count = delivered_count - returned_count

total_customers = customers["customer_id"].nunique()
purchasing_cust = order_summary["customer_id"].nunique()
purchase_rate   = purchasing_cust / total_customers * 100
dropoff         = 100 - purchase_rate
return_rate_pct = returned_count / total_orders * 100

metrics = dict(
    total_orders=total_orders, delivered_count=delivered_count,
    returned_count=returned_count, completed_count=completed_count,
    purchase_rate=purchase_rate, dropoff=dropoff,
    total_customers=total_customers, purchasing_cust=purchasing_cust,
)

print("=" * 55)
print("  KEY METRICS")
print("=" * 55)
print(f"  Total Customers        : {total_customers:,}")
print(f"  Purchasing Customers   : {purchasing_cust:,}")
print(f"  Customer Purchase Rate : {purchase_rate:.2f}%")
print(f"  Drop-off (no purchase) : {dropoff:.2f}%")
print(f"  Total Orders           : {total_orders:,}")
print(f"  Delivered Orders       : {delivered_count:,}")
print(f"  Returned Orders        : {returned_count:,}")
print(f"  Completed Orders       : {completed_count:,}")
print(f"  Order Return Rate      : {return_rate_pct:.2f}%")
print("=" * 55)


## 🔽 Order Conversion Funnel

In [ ]:
funnel = pd.DataFrame({
    "Stage": ["Total Orders", "Delivered", "Completed (no return)", "Returned"],
    "Count": [total_orders, delivered_count, completed_count, returned_count],
})
funnel["Conversion Rate (%)"] = (funnel["Count"] / total_orders * 100).round(2)
display(funnel)

PALETTE_FUNNEL = ["#3498db", "#2ecc71", "#27ae60", "#e74c3c"]
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(funnel["Stage"], funnel["Count"], color=PALETTE_FUNNEL, width=0.55)

for bar, (_, row) in zip(bars, funnel.iterrows()):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + total_orders * 0.01,
        f"{row['Count']:,}\n({row['Conversion Rate (%)']}%)",
        ha="center", va="bottom", fontsize=9, fontweight="bold",
    )

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.set_title("Order Conversion Funnel (End-to-End)")
ax.set_ylabel("Number of Orders")
ax.set_ylim(0, total_orders * 1.18)
plt.tight_layout()
plt.show()


## 🥧 Order Status Distribution

In [ ]:
status_counts = order_summary["order_status"].value_counts().reset_index()
status_counts.columns = ["Order Status", "Count"]
status_counts["%"] = (status_counts["Count"] / total_orders * 100).round(2)
display(status_counts)

fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax.pie(
    status_counts["Count"],
    labels=status_counts["Order Status"],
    autopct="%1.1f%%",
    startangle=140,
    colors=sns.color_palette("pastel")[: len(status_counts)],
)
for at in autotexts:
    at.set_fontsize(9)
ax.set_title("Order Status Distribution")
plt.tight_layout()
plt.show()


## ❌ Cancel Rate by Payment Method

In [ ]:
payment_cancel = (
    order_summary.groupby("payment_method")
    .agg(
        total_orders    = ("order_id", "nunique"),
        canceled_orders = ("order_status", lambda x: (x == "Canceled").sum()),
    )
    .reset_index()
)
payment_cancel["cancel_rate"] = (
    payment_cancel["canceled_orders"] / payment_cancel["total_orders"] * 100
).round(2)

payment_cancel = payment_cancel.sort_values("cancel_rate", ascending=False).reset_index(drop=True)
display(payment_cancel)

fig, ax = plt.subplots(figsize=(9, 5))
n    = len(payment_cancel)
bars = ax.bar(
    payment_cancel["payment_method"],
    payment_cancel["cancel_rate"],
    color=sns.color_palette("rocket", n),
    width=0.55,
)
max_rate = payment_cancel["cancel_rate"].max()
for bar, row in zip(bars, payment_cancel.itertuples()):   # FIX #2
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max_rate * 0.015,
        f"{row.cancel_rate}%",
        ha="center", va="bottom", fontsize=9,
    )
ax.set_title("Cancel Rate by Payment Method")
ax.set_ylabel("Cancel Rate (%)")
ax.set_ylim(0, max_rate * 1.2)
plt.tight_layout()
plt.show()


## 🔄 Return Rate by Category (Delivered Orders Only)

In [ ]:
delivered_cat = (
    oi.loc[oi["order_status"] == "Delivered", ["order_id", "category_name"]]
    .drop_duplicates()
    .merge(order_summary[["order_id", "is_returned"]], on="order_id", how="left")
)

category_return = (
    delivered_cat.groupby("category_name")
    .agg(
        total_orders    = ("order_id",    "nunique"),
        returned_orders = ("is_returned", "sum"),
    )
    .reset_index()
)
category_return["return_rate"] = (
    category_return["returned_orders"] / category_return["total_orders"] * 100
).round(2)

category_return = category_return.sort_values("return_rate", ascending=False).reset_index(drop=True)
display(category_return)

fig, ax = plt.subplots(figsize=(11, 5))
n    = len(category_return)
bars = ax.bar(
    category_return["category_name"],
    category_return["return_rate"],
    color=sns.color_palette("magma", n),
    width=0.6,
)
max_rate = category_return["return_rate"].max()
for bar, row in zip(bars, category_return.itertuples()):  # FIX #2
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max_rate * 0.015,
        f"{row.return_rate}%",
        ha="center", va="bottom", fontsize=8.5,
    )
ax.set_title("Return Rate by Product Category (Delivered Orders)")
ax.set_ylabel("Return Rate (%)")
ax.set_ylim(0, max_rate * 1.18)
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()


## 💰 Revenue by Category

In [ ]:
revenue_cat = (
    oi.groupby("category_name")
    .agg(total_revenue=("revenue", "sum"))
    .reset_index()
)
revenue_cat["revenue_pct"] = (
    revenue_cat["total_revenue"] / revenue_cat["total_revenue"].sum() * 100
).round(2)
revenue_cat = revenue_cat.sort_values("total_revenue", ascending=False).reset_index(drop=True)
display(revenue_cat)

fig, ax = plt.subplots(figsize=(9, 5))
n    = len(revenue_cat)
bars = ax.bar(
    revenue_cat["category_name"],
    revenue_cat["total_revenue"],
    color=sns.color_palette("viridis", n),
    width=0.6,
)
max_rev = revenue_cat["total_revenue"].max()
for bar, row in zip(bars, revenue_cat.itertuples()):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max_rev * 0.01,
        f"{row.revenue_pct}%",
        ha="center", va="bottom", fontsize=8.5,
    )
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M" if x >= 1e6 else f"{x/1e3:.0f}K")
)
ax.set_title("Total Revenue by Category")
ax.set_ylabel("Revenue")
ax.set_ylim(0, max_rev * 1.18)
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()


## 🛒 Average Order Value (AOV) by Category

In [ ]:
order_cat_rev = (
    oi.groupby(["order_id", "category_name"])["revenue"]
    .sum()
    .reset_index(name="order_total")
)
aov_cat = (
    order_cat_rev.groupby("category_name")["order_total"]
    .mean()
    .round(2)
    .reset_index(name="AOV")
    .sort_values("AOV", ascending=False)
    .reset_index(drop=True)
)
display(aov_cat)

fig, ax = plt.subplots(figsize=(9, 5))
n    = len(aov_cat)
bars = ax.bar(
    aov_cat["category_name"],
    aov_cat["AOV"],
    color=sns.color_palette("crest", n),
    width=0.6,
)
max_aov = aov_cat["AOV"].max()
for bar, row in zip(bars, aov_cat.itertuples()):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max_aov * 0.01,
        f"{row.AOV:,.0f}",
        ha="center", va="bottom", fontsize=8.5,
    )
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.set_title("Average Order Value (AOV) by Category")
ax.set_ylabel("AOV")
ax.set_ylim(0, max_aov * 1.18)
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()


## 🧪 A/B Test — Digital vs Non-Digital Payment Methods

**Hypothesis:** Do digital payment methods yield a higher delivery rate than non-digital / COD?


In [ ]:
digital_methods = {"Credit Card", "Debit Card", "E-Wallet", "Bank Transfer"}

order_summary["ab_group"] = order_summary["payment_method"].apply(
    lambda x: "A — Digital" if x in digital_methods else "B — Non-Digital / COD"
)

ab_metrics = (
    order_summary.groupby("ab_group")
    .agg(
        total_orders = ("order_id", "nunique"),
        delivered    = ("order_status", lambda x: (x == "Delivered").sum()),
    )
    .reset_index()
)
ab_metrics["delivery_rate (%)"] = (
    ab_metrics["delivered"] / ab_metrics["total_orders"] * 100
).round(2)
display(ab_metrics)

# Two-proportion z-test
if len(ab_metrics) >= 2:
    row_a = ab_metrics.iloc[0]
    row_b = ab_metrics.iloc[1]
    succ_A, n_A = int(row_a["delivered"]), int(row_a["total_orders"])
    succ_B, n_B = int(row_b["delivered"]), int(row_b["total_orders"])
    p_pool = (succ_A + succ_B) / (n_A + n_B)
    se     = np.sqrt(p_pool * (1 - p_pool) * (1 / n_A + 1 / n_B))
    z      = (succ_A / n_A - succ_B / n_B) / se
    p_val  = 2 * (1 - stats.norm.cdf(abs(z)))

    print(f"Z-score : {z:.4f}")
    print(f"p-value : {p_val:.4f}")
    alpha = 0.05
    if p_val < alpha:
        print(f"→ Statistically significant difference at α={alpha} (reject H₀)")
    else:
        print(f"→ No statistically significant difference at α={alpha} (fail to reject H₀)")

    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.bar(
        ab_metrics["ab_group"],
        ab_metrics["delivery_rate (%)"],
        color=["#3498db", "#e67e22"],
        width=0.45,
    )
    max_dr = ab_metrics["delivery_rate (%)"].max()
    for bar, row in zip(bars, ab_metrics.itertuples()):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max_dr * 0.01,
            f"{row._4}%\n(n={row.total_orders:,})",
            ha="center", va="bottom", fontsize=9,
        )
    ax.set_title(f"Delivery Rate: Digital vs Non-Digital\n(p-value = {p_val:.4f})")
    ax.set_ylabel("Delivery Rate (%)")
    ax.set_ylim(0, max_dr * 1.22)
    plt.tight_layout()
    plt.show()


## 💡 Key Insights

In [ ]:
top_rev    = revenue_cat.loc[revenue_cat["total_revenue"].idxmax()]
top_cancel = payment_cancel.loc[payment_cancel["cancel_rate"].idxmax()]
top_ret    = category_return.loc[category_return["return_rate"].idxmax()]
top_aov    = aov_cat.loc[aov_cat["AOV"].idxmax()]
funnel_dropoff = 100 - delivered_count / total_orders * 100

print("=" * 65)
print("  KEY INSIGHTS (BASED ON ACTUAL DATA)")
print("=" * 65)
print(f"1. Revenue Leader      : {top_rev['category_name']} ({top_rev['revenue_pct']}% revenue share)")
print(f"2. Cancel Rate Hotspot : {top_cancel['payment_method']} — highest cancel rate ({top_cancel['cancel_rate']:.1f}%)")
print(f"3. Highest Return Rate : {top_ret['category_name']} ({top_ret['return_rate']:.1f}%)")
print(f"4. Top AOV Category    : {top_aov['category_name']} (AOV = {top_aov['AOV']:,.0f})")
print(f"5. Funnel Drop-off     : {funnel_dropoff:.1f}% of orders did not reach Delivered status")
print(f"6. Customer Conversion : {purchase_rate:.1f}% of registered customers have placed an order")
print("=" * 65)
